In [1]:
import pandas as pd 

In [4]:

###### Step 1: Prepare the dataset (inheriting steps 1 & 2)
df = pd.read_csv('Ecommerce_Transactions_Cleaned.csv')
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Cohort_Month'] = df.groupby('Customer_ID')['Purchase_Date'].transform('min').dt.to_period('M')
df['Purchase_Month'] = df['Purchase_Date'].dt.to_period('M')
df['Cohort_Index'] = (df['Purchase_Month'].dt.year - df['Cohort_Month'].dt.year) * 12 + \
                     (df['Purchase_Month'].dt.month - df['Cohort_Month'].dt.month)

# Step 2: Group by Cohort Month & Index, then count unique active customers
cohort_counts = df.groupby(['Cohort_Month', 'Cohort_Index'])['Customer_ID'].nunique().reset_index()
cohort_counts.rename(columns={'Customer_ID': 'Active_Users'}, inplace=True)

# Step 3: Pivot the table to create a classic cohort matrix view
cohort_matrix = cohort_counts.pivot(index='Cohort_Month', columns='Cohort_Index', values='Active_Users')
cohort_matrix = cohort_matrix.fillna(0).astype(int)

print(cohort_matrix)

Cohort_Index   0   1   2   3   4  5  6  7
Cohort_Month                             
2025-01       66  27  26  12   4  1  1  0
2025-02       73  37  19  13   2  0  0  0
2025-03       76  37  31  17   9  3  0  0
2025-04       70  33  25   6   1  1  0  0
2025-05       53  24  14   9   4  0  0  0
2025-06       58  34  25  16  10  4  1  1
2025-07       58  31  13  10   6  4  0  0
2025-08       69  29  19  10   3  1  0  0
2025-09       81  40  30  16   4  4  0  0
2025-10       64  35  18   9   1  2  0  0
2025-11       61  28  17  10   6  1  0  0
2025-12       64  25  14   6   2  0  0  0
2026-01        7   1   0   0   0  0  0  0


In [5]:
# Filter successful transactions
df_success = df[df['Payment_Status'] == 'Success']

# Aggregate total revenue generated per cohort index
revenue_matrix = df_success.groupby(['Cohort_Month', 'Cohort_Index'])[
    'Total_Amount'
].sum().unstack()

# Compute Average Revenue Per User (ARPU)
user_matrix = df_success.groupby(['Cohort_Month', 'Cohort_Index'])[
    'Customer_ID'
].nunique().unstack()
arpu_matrix = revenue_matrix / user_matrix

print("--- Total Revenue Matrix ($) ---")
display(revenue_matrix.fillna(0))

--- Total Revenue Matrix ($) ---


Cohort_Index,0,1,2,3,4,5,6
Cohort_Month,,,,,,,
2025-01,288961.0,140678.0,168380.0,57510.0,13433.0,2999.0,799.0
2025-02,350830.0,213759.0,84317.0,69831.0,18995.0,0.0,0.0
2025-03,328836.0,238764.0,142225.0,107195.0,74422.0,14481.0,0.0
2025-04,286550.0,188787.0,127312.0,53574.0,3995.0,5997.0,0.0
2025-05,217088.0,131417.0,80746.0,33048.0,10491.0,0.0,0.0
2025-06,303269.0,195523.0,150956.0,145436.0,110133.0,46243.0,1719.0
2025-07,289516.0,157033.0,70031.0,91689.0,54457.0,19289.0,0.0
2025-08,398629.0,209153.0,135234.0,55955.0,23260.0,15796.0,0.0
2025-09,406050.0,239687.0,217015.0,66418.0,25660.0,34139.0,0.0
